# CoShop User Simulator Explorer

This notebook demonstrates how to use the `GatedFeatureUser` simulator — a synthetic user
that responds to a shopping assistant's messages based on a ground-truth preference specification.

The simulator implements the *gated feature* model: it starts with only some features known
and reveals more preferences as the conversation progresses.

In [1]:
%load_ext autoreload
%autoreload 2

import os
# Set your LLM provider API key before running:
# os.environ["ANTHROPIC_API_KEY"] = "..."
# os.environ["OPENAI_API_KEY"] = "..."

In [4]:
from coshop.data import get_dataset

# Choose a dataset: 'movielens', 'goodreads', or 'hm'
DATASET = "hm"
SPEC_IDX = 0  # which spec (user) to simulate

ds = get_dataset(DATASET, dev=False, max_xstar=1)
spec = ds[SPEC_IDX]

print(f"Dataset: {DATASET}  |  Item type: {spec.item_name}")
print(f"Catalog size: {len(ds.catalog)}")
print(f"\nGround-truth target items (xstar): {spec.xstar}")

/lfs/guestrin-hgx-1/0/irena/elicitation-v2/clean_repo/coshop/data/hm/data.py:132: DtypeWarning: Columns (75) have mixed types. Specify dtype option on import or set low_memory=False.
  catalog_df = pd.read_csv(catalog_path, index_col="id")


ustar not available (vector search unreachable): Vector search API URL not configured. Pass api_url= at construction or set the VECTOR_SEARCH_API_URL environment variable.
Dataset: hm  |  Item type: clothing item
Catalog size: 37570

Ground-truth target items (xstar): ['903326005']


In [5]:
# Inspect the ground-truth preferences (simulator view of the target items)
spec.xstar_simulator_view

,product_type_name,graphical_appearance_name,colour_group_name,perceived_colour_master_name,price,closure,fit,material,sleeve_length,neckline,...,seam_construction_type,has_yoke_panel,is_wrap_style,garment_length_category,vegan,toxin_free,nursing_access_design,gender,age,is_sport
id,,,,,,,,,,,,,,,,,,,,,
903326005,T-shirt,Placement print,Light Beige,Beige,<= $30,open to anything,relaxed,cotton,short,crew/round neck,...,open to anything,False,False,open to anything,False,False,open to anything,male,adult,False


## 2. Initialize the user simulator

In [ ]:
from coshop.user_simulator import get_simulator

# MODEL_NAME can be any LangChain-compatible model string, e.g.:
#   "claude-haiku-4-5"  (requires ANTHROPIC_API_KEY)
#   "gpt-4o-mini"          (requires OPENAI_API_KEY)
MODEL_NAME = "gpt-5-nano"

sim = get_simulator(
    "copref_user",
    spec=spec,
    dataset=ds,
    model_name=MODEL_NAME,
    verbosity=1,       # 0=silent, 1=outputs only, 2=full prompts
    seed=42,
)
print("Simulator initialized.")

Error getting history search tool
Model kwargs:  {}
MESSAGE PARSER INITIALIZED
Use item jsons: False
Use oracle item representations: False
Use actual item values: False
Max text len: None
[FeatureTracker.__init__] search_features: ['product_type_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_master_name', 'price', 'closure', 'fit', 'material', 'sleeve_length', 'neckline', 'has_hood', 'has_lining', 'interior_finish', 'has_pockets', 'has_ribbing', 'has_drawstring', 'is_sustainable', 'formality', 'target_lifestyle_style', 'waist_style', 'waist_treatment', 'waist_styling', 'leg_style', 'pant_leg_length', 'fly_type', 'belt_loop_presence', 'crotch_structure', 'has_cargo_pockets', 'dress_length', 'skirt_shape', 'skirt_pleating', 'bodice_style', 'leg_structure', 'pack_size_configuration', 'top_garment_type', 'bottom_garment_type', 'sleeve_style', 'sleeve_cuff_detail', 'sleeve_graphics', 'shoulder_style', 'shoulder_exposure_style', 'jacket_type', 'has_detachable_hoo

## 3. Run the conversation

Call `sim(assistant_message)` to get the user's response.
Call `sim()` or `sim("")` to let the user speak first (opening message).

In [8]:
# Opening: assistant greets the user
user_reply = sim("Hi! I'm a shopping assistant. What are you looking for today?")
print("USER:", user_reply)

Model kwargs:  {}
Any system message in state: False
Current chain length: 1. Forcing chain to continue: False
[MessageParser._parse_items] Completed with zero items; item_ids= dict_keys([])
Parsing questions; allowed cols = ['product_type_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_master_name', 'price', 'closure', 'fit', 'material', 'sleeve_length', 'neckline', 'has_hood', 'has_lining', 'interior_finish', 'has_pockets', 'has_ribbing', 'has_drawstring', 'is_sustainable', 'formality', 'target_lifestyle_style', 'waist_style', 'waist_treatment', 'waist_styling', 'leg_style', 'pant_leg_length', 'fly_type', 'belt_loop_presence', 'crotch_structure', 'has_cargo_pockets', 'dress_length', 'skirt_shape', 'skirt_pleating', 'bodice_style', 'leg_structure', 'pack_size_configuration', 'top_garment_type', 'bottom_garment_type', 'sleeve_style', 'sleeve_cuff_detail', 'sleeve_graphics', 'shoulder_style', 'shoulder_exposure_style', 'jacket_type', 'has_detachable_hood', 'po

Mapping: 100%|██████████ 1/1 LM calls [00:04<00:00,  4.48s/it]
Filtering: 100%|██████████ 77/77 LM calls [00:26<00:00,  2.90it/s]


What are you looking for today? ['product_type_name', 'closure', 'fit', 'sleeve_length', 'neckline', 'interior_finish', 'has_ribbing', 'target_lifestyle_style', 'waist_style', 'waist_treatment', 'waist_styling', 'leg_style', 'pant_leg_length', 'belt_loop_presence', 'crotch_structure', 'dress_length', 'skirt_pleating', 'bodice_style', 'leg_structure', 'top_garment_type', 'bottom_garment_type', 'sleeve_style', 'sleeve_cuff_detail', 'sleeve_graphics', 'shoulder_style', 'jacket_type', 'has_detachable_hood', 'pocket_type', 'pocket_location', 'fabric_surface_effect', 'surface_texture', 'functional_finish', 'waterproof', 'windproof', 'pattern_type', 'graphic_theme', 'graphic_placement', 'decorative_trim_type', 'has_logo_or_brand_mark', 'silhouette_shape', 'strap_configuration', 'hem_detail', 'back_detailing', 'side_features', 'seam_construction_type', 'gender', 'is_sport']


Mapping: 100%|██████████ 1/1 LM calls [00:03<00:00,  3.22s/it]


[GatedFeatureUser._generate_response] parsed_message: ParsedMessage(items_to_evaluate=[], clarifying_questions=[ClarifyingQuestion(question='What are you looking for today?', relevant_columns=['product_type_name', 'closure', 'fit'])], explanations=[])
['closure', 'fit']


Mapping: 100%|██████████ 1/1 LM calls [00:03<00:00,  3.48s/it]


[FeatureTracker._reveal_feature] Revealing feature closure from question
[FeatureTracker._reveal_feature] Revealing feature fit from question


Mapping: 100%|██████████ 1/1 LM calls [00:07<00:00,  7.25s/it]

[__call__] Runtime cost: 0.0s, Token cost: 0
USER: *What are you looking for today?*: I'm after a relaxed-fit T-shirt, and I'm open to any closure. For the feature hint, I don't know yet.


In [ ]:
# Assistant asks clarifying questions
assistant_msg = (
    "Great, I'd love to help! To find the best match, could you tell me:\n"
    "1. Do you have a preferred color or pattern?\n"
    "2. What's your preferred fit (regular, loose, fitted)?\n"
    "3. Do you have a budget in mind?"
)
user_reply = sim(assistant_msg)
print("USER:", user_reply)

In [ ]:
# Continue the conversation
user_reply = sim(
    "Thanks! Based on that I have a few options. "
    "Are there any materials you prefer or want to avoid?"
)
print("USER:", user_reply)

## 4. Inspect the message parser

The `MessageParser` converts each assistant message into structured actions
(clarifying questions, item feedback requests, explanations).

In [ ]:
# What actions did the parser see in the last turn?
if sim.message_parser.parsed_history:
    last = sim.message_parser.parsed_history[-1]
    print("Clarifying questions detected:", last.clarifying_questions)
    print("Items to evaluate:", last.items_to_evaluate)

In [ ]:
# Full parse history (one entry per turn)
sim.message_parser.parsed_history

In [ ]:
# Current feature summary (what the parser has learned so far)
sim.message_parser._feature_summary

## 5. Inspect the feature tracker

The `FeatureTracker` keeps track of which features the simulator has revealed
vs. which are still hidden.

In [ ]:
# Features revealed so far (known to the policy)
print("Known context:")
print(sim.feature_tracker.get_known_context())

In [ ]:
# Features still hidden (not yet revealed)
print("Unknown search features:", sim.feature_tracker.unknown_search_features)
print("Unknown experience features:", sim.feature_tracker.unknown_experience_features)
print("Unknown credence features:", sim.feature_tracker.unknown_credence_features)

In [ ]:
# Reveal history — which features were revealed and when
sim.feature_tracker.reveal_history

## 6. Test structured item feedback

Use `SHOW_ITEM_FOR_FEEDBACK` to ask the simulator to react to a specific item.
The simulator will compare the item's features against the user's (hidden) preferences.

In [ ]:
# Pick a catalog item to show
sample_id = str(ds.catalog.index[0])
sample_row = ds.catalog.loc[sample_id]
print(f"Showing item {sample_id}: {sample_row.get('prod_name', sample_row.get('title', sample_id))}")

# Format as a SHOW_ITEM_FOR_FEEDBACK action
import json
action = f'SHOW_ITEM_FOR_FEEDBACK {{"item_id": "{sample_id}"}}'
parsed = sim.message_parser.parse(action)
print("\nParsed action:", parsed)

In [ ]:
# Get the simulator's feedback on the item
user_reply = sim(action)
print("USER:", user_reply)

## 7. Rank items (simulator's internal ranking)

`sim.rank_items` asks the simulator to rank a small set of items based on what it knows.

In [ ]:
# Pick a handful of catalog items to rank
sample_ids = [str(i) for i in ds.catalog.index[:5]]
items = {
    item_id: ds.simulator_catalog.loc[item_id].to_json()
    for item_id in sample_ids
    if item_id in ds.simulator_catalog.index
}

ranked = sim.rank_items(items, mode="parser")
print("Ranked items (best to worst):", ranked)

## 8. Inspect full conversation history

In [ ]:
# Conversation turns so far (each turn is a ConversationTurn with .msg, .actions)
for i, turn in enumerate(sim.conversation_history):
    msg = turn.msg[:200] + ("..." if len(turn.msg) > 200 else "")
    print(f"[Turn {i}] {msg}\n")

In [ ]:
# Full internal state snapshots (message parser, feature tracker, item comparer per turn)
state_history = sim.get_state_history()
print("State history keys:", list(state_history.keys()))

## 9. Reset and try a different spec

In [ ]:
# Reset the simulator and load a different spec
new_spec = ds[1]
sim.reset(spec=new_spec)
print(f"New xstar: {new_spec.xstar}")
print(f"New preferences: {new_spec.xstar_simulator_view.to_dict()}")

reply = sim("Hello! What kind of item are you looking for?")
print("USER:", reply)